# Part 0 — Meet SmolVLM

Before we wrap anything in an API, let's load the model itself, talk to it directly, and end up with two small functions — `load_model()` and `generate()` — that we'll drop straight into `api/model.py` in the next part.

This notebook is designed to run on **Google Colab** — a free Python kernel with most ML libraries pre-installed, no local setup required. The cell below upgrades `transformers` to a version that knows about SmolVLM; everything after that is pure model code. The first `from_pretrained` call will download `SmolVLM-256M-Instruct` (~500 MB) from the Hugging Face Hub into Colab's session cache.

When you're done, you'll copy the final two functions (`load_model` and `generate`) into your local clone of the repo for the API to use.

In [ ]:
# Colab bootstrap — Colab's pre-installed `transformers` is sometimes a bit
# old for SmolVLM. Run this once; it's a near-instant no-op if a recent
# enough version is already on the kernel.
%pip install -q "transformers>=4.47,<4.50"

In [ ]:
import torch
from PIL import Image
from transformers import AutoModelForVision2Seq, AutoProcessor

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"

if torch.cuda.is_available():
    device, dtype = "cuda", torch.bfloat16     # GPU: smaller dtype, faster
else:
    device, dtype = "cpu", torch.float32       # CPU: float32 is the safe default

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()    # disable dropout — we're only doing inference

print(f"Loaded {MODEL_ID} on {device}")

## 1. Why the *Instruct* variant?

`SmolVLM-256M-Instruct` is an **instruction-tuned** variant of the base SmolVLM.

* The *base* model was pre-trained to predict the next token on raw text-and-image data — it can complete a sentence, but it has no notion of a "conversation".
* The *instruct* version was further fine-tuned on conversational examples, so it knows how to read a back-and-forth dialogue between **user** and **assistant**, and produce an appropriate assistant reply.

That fine-tuning is exactly why we can hand it a structured list of messages and get a sensible answer back.

## 2. The shape of a conversation

A conversation is a list of **messages**. Each message has a `role` and a `content`:

| Role        | Who          | Notes                                                                 |
|-------------|--------------|-----------------------------------------------------------------------|
| `system`    | Instructions | Optional, at most one, at the very start. Sets persona / tone.        |
| `user`      | The human    | What the human says.                                                  |
| `assistant` | The model    | What the model said in previous turns of this same conversation.      |

The model is trained to read this structure and pay attention to the role boundaries when generating the next assistant reply.

We're going to keep things deliberately simple: each `content` is just a string. This is the same format the OpenAI chat API uses — call it our **lingua franca**. Hold on to this; it'll matter again in §6.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Keep replies short."},
    {"role": "user",   "content": "What is a vision-language model, in one sentence?"},
]
messages

## 3. From `{role, content}` to SmolVLM's content-block format

SmolVLM doesn't take our simple shape directly. Internally, it expects each message's `content` to be a **list of content blocks** — each block tagged with a type (`text` or `image`):

```python
{"role": "user", "content": [
    {"type": "text", "text": "Hello!"},
]}
```

This is what `processor.apply_chat_template(...)` consumes. Let's do the conversion by hand once so we can see exactly what's happening — afterwards we'll bury it inside our `generate()` function and never look at it again.

In [ ]:
def to_content_blocks(messages):
    """Convert {role, content: str} → {role, content: [{type:text, text: ...}]} for SmolVLM."""
    return [
        {"role": m["role"], "content": [{"type": "text", "text": m["content"]}]}
        for m in messages
    ]

formatted = to_content_blocks(messages)
prompt = processor.apply_chat_template(formatted, add_generation_prompt=True)

print(prompt[:500])
print("...")

## 4. First generation

Tokenize the prompt, run `model.generate`, and slice off the **input tokens** so we only decode the new ones.

`do_sample=False` (equivalently `temperature=0`) gives a greedy reply — same input always produces the same output. `do_sample=True` samples from the distribution and gives variation.

In [ ]:
inputs = processor(text=prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,    # deterministic
    )

# model.generate returns [input_tokens, new_tokens]; we only want the new ones.
input_len = inputs["input_ids"].shape[1]
reply = processor.decode(output_ids[0, input_len:], skip_special_tokens=True).strip()
print(reply)

## 5. Adding an image

To ask a question *about* an image, we attach an `{"type": "image"}` block to the **last user message** (alongside the text block), and pass the actual PIL image to the processor separately.

Two details worth pinning down:

* The image placeholder is **only** on the *last* user message — earlier turns might reference the image in their text, but they don't carry the image block.
* The chat template doesn't embed image bytes; it inserts a special token where the image goes. The processor handles the actual image encoding when we pass `images=[...]`.

In [ ]:
from PIL import ImageDraw

# Synthetic image: a red square on a white background
img = Image.new("RGB", (224, 224), "white")
ImageDraw.Draw(img).rectangle([60, 60, 164, 164], fill="red")
img

In [ ]:
messages_with_image = [
    {"role": "user", "content": "What do you see in this image? One sentence."},
]


def to_content_blocks_with_image(messages, has_image):
    """Same as to_content_blocks, but attach the image block to the LAST user message."""
    last_user_idx = max(
        (i for i, m in enumerate(messages) if m["role"] == "user"), default=-1
    )
    out = []
    for i, m in enumerate(messages):
        if m["role"] == "user" and i == last_user_idx and has_image:
            content = [{"type": "image"}, {"type": "text", "text": m["content"]}]
        else:
            content = [{"type": "text", "text": m["content"]}]
        out.append({"role": m["role"], "content": content})
    return out


formatted = to_content_blocks_with_image(messages_with_image, has_image=True)
prompt = processor.apply_chat_template(formatted, add_generation_prompt=True)

inputs = processor(text=prompt, images=[img], return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=60, do_sample=False)

input_len = inputs["input_ids"].shape[1]
reply = processor.decode(output_ids[0, input_len:], skip_special_tokens=True).strip()
print(reply)

## 6. Factor it out — `load_model()` and `generate()`

We now have all the pieces. Let's collapse them into two functions we can drop into `api/model.py` as-is.

### A design note before we write them

Our `generate()` will accept the **simple `{role, content}` format** — the same format we typed by hand in §2, and the same format the OpenAI chat API uses. The conversion to SmolVLM's content-block format will live **inside** `generate()`, hidden from anyone who calls it.

Why call this out? Because in the next part, when we put the model behind an HTTP API, the API contract will *also* use `{role, content}`. By picking that as our lingua franca on both sides:

* **The API never has to translate.** Clients send `{role, content}`, the API forwards it to `generate()` essentially as-is — at most a one-line Pydantic-to-dict unpack.
* **`model.py` is the only file that knows what a content block is.** That knowledge is a SmolVLM implementation detail; if we ever swap to a different VLM, only `model.py` changes — `schemas.py`, `main.py` and the UI stay put.
* **No `format_messages` helper sitting halfway between two files.** A translator between two formats we control on both sides would be pure indirection — so we don't write one.

In [ ]:
# These two functions are exactly what `api/model.py` will contain.

_processor = None
_model = None
_device = "cpu"


def load_model():
    """Download (first time) and load SmolVLM into memory. Idempotent."""
    global _processor, _model, _device
    if _model is not None:
        return                                             # already loaded
    if torch.cuda.is_available():
        _device, dtype = "cuda", torch.bfloat16
    else:
        _device, dtype = "cpu", torch.float32
    _processor = AutoProcessor.from_pretrained(MODEL_ID)
    _model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, torch_dtype=dtype).to(_device)
    _model.eval()


def generate(messages, image=None, max_new_tokens=256, temperature=0.7):
    """Generate the next assistant reply for a conversation.

    Args:
        messages: list of {'role': 'system'|'user'|'assistant', 'content': str}.
        image:    optional PIL.Image attached to the LAST user message.
        max_new_tokens: generation budget.
        temperature:   0 for greedy, >0 for sampling.

    Returns:
        The decoded assistant reply as a string.
    """
    has_image = image is not None
    last_user_idx = max(
        (i for i, m in enumerate(messages) if m["role"] == "user"), default=-1
    )

    # SmolVLM-specific: convert {role, content: str} into the content-block format.
    formatted = []
    for i, m in enumerate(messages):
        if m["role"] == "user" and i == last_user_idx and has_image:
            content = [{"type": "image"}, {"type": "text", "text": m["content"]}]
        else:
            content = [{"type": "text", "text": m["content"]}]
        formatted.append({"role": m["role"], "content": content})

    prompt = _processor.apply_chat_template(formatted, add_generation_prompt=True)
    inputs = _processor(
        text=prompt,
        images=[image] if has_image else None,
        return_tensors="pt",
    ).to(_device)

    do_sample = temperature > 0.0
    with torch.no_grad():
        output_ids = _model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
        )
    input_len = inputs["input_ids"].shape[1]
    return _processor.decode(output_ids[0, input_len:], skip_special_tokens=True).strip()

## 7. Sanity check — call the functions we just wrote

In [ ]:
load_model()    # no-op if already loaded above

# Text-only
print(generate(
    [{"role": "user", "content": "Say hi in three words."}],
    max_new_tokens=20,
    temperature=0.0,
))

# Text + image
print(generate(
    [{"role": "user", "content": "What color is this image, briefly?"}],
    image=img,
    max_new_tokens=20,
    temperature=0.0,
))

## 8. What's next

Two functions, two responsibilities:

* `load_model()` — runs once, downloads + warms up the model.
* `generate(messages, image, max_new_tokens, temperature)` — runs on every chat request.

In Part 1 of `vlm_chatbot.md`, we'll drop both functions verbatim into `api/model.py`, design the HTTP contract around them, and wrap everything in FastAPI.